# ¿Qué es una API? Ejemplos con PokeAPI y Mastodon

Una **API** (*Application Programming Interface*) es una forma acordada en que un programa le pide datos o acciones a otro programa.

En la web, la mayoría de las APIs funcionan así:

1. Nuestro programa (el **cliente**) envía una **petición** (*request*) a una dirección (URL), llamada **endpoint**.
2. El **servidor** responde con un **código de estado** (por ejemplo, `200` = todo bien, `404` = no encontrado) y un **cuerpo** con los datos.
3. Casi siempre los datos vienen en formato **JSON**, que en Python se convierte en diccionarios y listas, estructuras que ya conocemos.

En este cuaderno:

- **Parte 1:** usamos la [PokeAPI](https://pokeapi.co/) para entender la mecánica básica (no requiere registro).
- **Parte 2:** usamos la API de [Mastodon](https://docs.joinmastodon.org/api/) para obtener publicaciones reales y analizarlas como texto.

## 0. Preparación

Usaremos la biblioteca `requests`, el estándar en Python para hacer peticiones HTTP. Si no la tienes instalada, descomenta la siguiente línea.

In [ ]:
# !pip install requests

In [ ]:
import requests

---
# Parte 1: PokeAPI

## 1.1 La primera petición

La URL base de la PokeAPI es `https://pokeapi.co/api/v2/`. A ella le agregamos el **recurso** que queremos (`pokemon`) y el **identificador** (un nombre o un número).

In [ ]:
url = "https://pokeapi.co/api/v2/pokemon/pikachu"
respuesta = requests.get(url)

print(respuesta.status_code)                  # 200 significa éxito
print(respuesta.headers["Content-Type"])      # tipo de contenido de la respuesta

## 1.2 De JSON a diccionario

El método `.json()` convierte el texto de la respuesta en un diccionario de Python. Veamos qué **llaves** tiene.

In [ ]:
datos = respuesta.json()

print(type(datos))
print(list(datos.keys()))

Ahora podemos navegar el diccionario como cualquier otro: con corchetes para las llaves y con índices para las listas.

In [ ]:
print("Nombre:", datos["name"])
print("Número:", datos["id"])
print("Altura (decímetros):", datos["height"])
print("Peso (hectogramos):", datos["weight"])

In [ ]:
# 'types' es una lista de diccionarios anidados
for t in datos["types"]:
    print(t["slot"], t["type"]["name"])

In [ ]:
# Comprensión de listas para extraer solo los nombres de las habilidades
habilidades = [h["ability"]["name"] for h in datos["abilities"]]
habilidades

## 1.3 Errores: ¿qué pasa si pedimos algo que no existe?

Siempre hay que revisar el código de estado antes de usar los datos.

In [ ]:
respuesta = requests.get("https://pokeapi.co/api/v2/pokemon/pokemon-que-no-existe")
print(respuesta.status_code)

## 1.4 Envolver la petición en una función

Es buena práctica encapsular la lógica en una función que maneje los errores.

In [ ]:
BASE_POKEAPI = "https://pokeapi.co/api/v2"

def obtener_pokemon(nombre):
    """Regresa un diccionario con datos básicos del pokémon, o None si no existe."""
    respuesta = requests.get(f"{BASE_POKEAPI}/pokemon/{nombre.lower()}", timeout=10)
    if respuesta.status_code != 200:
        return None
    datos = respuesta.json()
    return {
        "nombre": datos["name"],
        "tipos": [t["type"]["name"] for t in datos["types"]],
        "altura_m": datos["height"] / 10,
        "peso_kg": datos["weight"] / 10,
    }

for nombre in ["Bulbasaur", "Charmander", "Squirtle", "Missingno"]:
    print(nombre, "->", obtener_pokemon(nombre))

## 1.5 Parámetros de consulta (*query parameters*)

Muchos endpoints aceptan parámetros después de `?` en la URL, por ejemplo `?limit=5&offset=0`. Con `requests` los pasamos como diccionario en `params`, y la biblioteca construye la URL por nosotros. Esto sirve para la **paginación**: pedir los resultados por partes.

In [ ]:
respuesta = requests.get(f"{BASE_POKEAPI}/pokemon", params={"limit": 5, "offset": 0}, timeout=10)
print(respuesta.url)   # la URL que se construyó

lista = respuesta.json()
print("Total de pokémon:", lista["count"])
for p in lista["results"]:
    print(p["name"], p["url"])

## 1.6 Un ejemplo lingüístico: nombres en distintos idiomas

El recurso `pokemon-species` incluye los nombres del pokémon en varios idiomas, cada uno etiquetado con su código de idioma.

In [ ]:
especie = requests.get(f"{BASE_POKEAPI}/pokemon-species/pikachu", timeout=10).json()

for n in especie["names"]:
    print(f'{n["language"]["name"]:>8}: {n["name"]}')

In [ ]:
# Las descripciones (flavor text) también están en varios idiomas; filtremos las del español
descripciones_es = {
    d["flavor_text"].replace("\n", " ").replace("\f", " ")
    for d in especie["flavor_text_entries"]
    if d["language"]["name"] == "es"
}
for d in descripciones_es:
    print("-", d)

---
# Parte 2: Mastodon

## ¿Qué es Mastodon?

[Mastodon](https://joinmastodon.org/) es una red social de *microblogging* (publicaciones breves, parecida a X/Twitter), creada en 2016 por Eugen Rochko. Es **software libre** y la desarrolla una organización sin fines de lucro. Sus rasgos principales son:

- **Descentralizada:** no hay una sola empresa ni un solo servidor. Hay miles de servidores independientes, llamados **instancias** (por ejemplo, `mastodon.social`, `lingo.lol` o `mastodon.uy`), y cada una tiene sus propias reglas y moderación.
- **Federada:** las instancias se comunican entre sí mediante el protocolo abierto **ActivityPub**. Una persona con cuenta en una instancia puede seguir a alguien de otra. Al conjunto de servidores conectados se le llama el **Fediverso**.
- **Cuentas con dominio:** por eso una cuenta se escribe como un correo electrónico, `@usuario@instancia`.
- **Línea de tiempo cronológica:** no hay un algoritmo que decida qué ves. Las publicaciones (*toots* o *statuses*) se ordenan por fecha y se pueden etiquetar con su **idioma**, lo cual es muy útil para quienes estudiamos lenguaje.

## La API de Mastodon

Todas las instancias exponen la misma API, así que el código de esta sección sirve para cualquiera de ellas con solo cambiar la URL.

Algunos endpoints son públicos (no requieren cuenta) y otros requieren un **token de acceso** (por ejemplo, para publicar). Empezaremos con los públicos.

Documentación: <https://docs.joinmastodon.org/methods/>

> **Nota sobre las instancias:** cada instancia decide si permite leer su API sin autenticación. En `mastodon.social` todos los ejemplos de lectura funcionan sin token, pero algunas instancias exigen token incluso para consultar líneas de tiempo públicas o etiquetas. Si cambias `INSTANCIA` por otro servidor y obtienes un error `401` (no autorizado) o `422`, vuelve a `mastodon.social` o envía tu token también en las peticiones `GET` (ver la sección 2.7).

In [ ]:
INSTANCIA = "https://mastodon.social"

info = requests.get(f"{INSTANCIA}/api/v2/instance", timeout=10).json()
print("Instancia:", info["domain"])
print("Descripción:", info["description"])
print("Idiomas:", info["languages"])

## 2.1 Publicaciones con una etiqueta (*hashtag*)

El endpoint `/api/v1/timelines/tag/{etiqueta}` regresa las publicaciones (en Mastodon se llaman *statuses*) más recientes con esa etiqueta. La respuesta es una **lista** de diccionarios.

In [ ]:
respuesta = requests.get(
    f"{INSTANCIA}/api/v1/timelines/tag/linguistics",
    params={"limit": 20},
    timeout=10,
)
print(respuesta.status_code)

publicaciones = respuesta.json()
print("Publicaciones recibidas:", len(publicaciones))
print("Llaves de cada publicación:", list(publicaciones[0].keys()))

In [ ]:
p = publicaciones[0]
print("Autor:", p["account"]["acct"])
print("Fecha:", p["created_at"])
print("Idioma:", p["language"])
print("URL:", p["url"])
print("Contenido (crudo):", p["content"][:300])

## 2.2 Limpiar el HTML

El campo `content` viene en **HTML** (con etiquetas como `<p>` y `<a>`). Para analizar el texto necesitamos quitarlas. Usaremos el módulo `html.parser` de la biblioteca estándar.

In [ ]:
from html.parser import HTMLParser

class ExtractorDeTexto(HTMLParser):
    """Junta solo el texto de un documento HTML, ignorando las etiquetas."""
    def __init__(self):
        super().__init__()
        self.partes = []

    def handle_starttag(self, tag, attrs):
        if tag in ("br", "p"):
            self.partes.append(" ")

    def handle_data(self, data):
        self.partes.append(data)

def html_a_texto(html):
    extractor = ExtractorDeTexto()
    extractor.feed(html)
    return " ".join("".join(extractor.partes).split())

print(html_a_texto(p["content"]))

## 2.3 Del JSON a una tabla

Juntemos lo que nos interesa de cada publicación en una lista de diccionarios más simple.

In [ ]:
def resumir(publicacion):
    return {
        "autor": publicacion["account"]["acct"],
        "fecha": publicacion["created_at"][:10],
        "idioma": publicacion["language"],
        "texto": html_a_texto(publicacion["content"]),
        "etiquetas": [t["name"] for t in publicacion["tags"]],
    }

corpus = [resumir(p) for p in publicaciones]

for pub in corpus[:5]:
    print(f'[{pub["fecha"]}] ({pub["idioma"]}) @{pub["autor"]}')
    print("   ", pub["texto"][:120])

## 2.4 Un poco de análisis

Ya tenemos un pequeño **corpus**. Usemos `collections.Counter` para contar idiomas, etiquetas y palabras.

In [ ]:
from collections import Counter

idiomas = Counter(pub["idioma"] for pub in corpus)
idiomas.most_common()

In [ ]:
etiquetas = Counter(e for pub in corpus for e in pub["etiquetas"])
etiquetas.most_common(10)

In [ ]:
import re

def tokenizar(texto):
    """Separa en palabras: quita URLs, menciones y etiquetas, y pasa a minúsculas."""
    texto = re.sub(r"https?://\S+|[@#]\w+", " ", texto)
    return re.findall(r"[^\W\d_]+", texto.lower())

palabras = [w for pub in corpus for w in tokenizar(pub["texto"])]
print("Tokens:", len(palabras))
print("Tipos (palabras distintas):", len(set(palabras)))
print("Proporción tipo/token:", round(len(set(palabras)) / len(palabras), 3))

In [ ]:
# Las palabras más frecuentes de más de 3 letras (un filtro muy simple de palabras vacías)
Counter(w for w in palabras if len(w) > 3).most_common(15)

## 2.5 Paginación en Mastodon

Mastodon no usa `offset`; usa el parámetro `max_id`: "dame publicaciones más antiguas que esta". Así podemos juntar más de una página.

In [ ]:
def publicaciones_por_etiqueta(etiqueta, paginas=3, por_pagina=40, instancia=INSTANCIA):
    resultado = []
    max_id = None
    for _ in range(paginas):
        params = {"limit": por_pagina}
        if max_id:
            params["max_id"] = max_id
        r = requests.get(f"{instancia}/api/v1/timelines/tag/{etiqueta}", params=params, timeout=10)
        r.raise_for_status()           # lanza una excepción si el código no es 2xx
        pagina = r.json()
        if not pagina:
            break
        resultado.extend(pagina)
        max_id = pagina[-1]["id"]      # la más antigua de esta página
    return resultado

lote = publicaciones_por_etiqueta("lingüística", paginas=2)
print("Publicaciones:", len(lote))
Counter(p["language"] for p in lote).most_common()

## 2.6 Buscar una cuenta y ver sus publicaciones

Con `/api/v1/accounts/lookup` obtenemos una cuenta por su nombre, y con su `id` pedimos sus publicaciones.

In [ ]:
cuenta = requests.get(
    f"{INSTANCIA}/api/v1/accounts/lookup",
    params={"acct": "Mastodon"},
    timeout=10,
).json()

print(cuenta["display_name"], "-", cuenta["url"])
print("Seguidores:", cuenta["followers_count"])

estados = requests.get(
    f"{INSTANCIA}/api/v1/accounts/{cuenta['id']}/statuses",
    params={"limit": 5, "exclude_replies": True, "exclude_reblogs": True},
    timeout=10,
).json()

for e in estados:
    print("-", e["created_at"][:10], html_a_texto(e["content"])[:100])

## 2.7 (Opcional) Publicar con un token de acceso

Para **escribir** (publicar, marcar favoritos, etc.) se necesita autenticación:

1. En tu instancia ve a **Preferencias → Desarrollo → Nueva aplicación**.
2. Dale permiso `write:statuses` y guarda.
3. Copia el **token de acceso**.

⚠️ **Nunca** escribas el token directamente en el cuaderno ni lo subas a GitHub: es como tu contraseña. Guárdalo en una variable de entorno (`MASTODON_TOKEN`).

El token se envía en el encabezado `Authorization`, y en lugar de `GET` usamos `POST`, porque estamos **creando** algo.

In [ ]:
import os

TOKEN = os.environ.get("MASTODON_TOKEN")

def publicar(texto, instancia=INSTANCIA, visibilidad="private"):
    """Publica un estado. 'private' lo hace visible solo para tus seguidores."""
    r = requests.post(
        f"{instancia}/api/v1/statuses",
        headers={"Authorization": f"Bearer {TOKEN}"},
        data={"status": texto, "visibility": visibilidad, "language": "es"},
        timeout=10,
    )
    r.raise_for_status()
    return r.json()["url"]

if TOKEN:
    print("Publicado en:", publicar("Hola desde Python 🐍 #ProgramaciónParaLingüistas"))
else:
    print("No hay token: define la variable de entorno MASTODON_TOKEN para publicar.")

---
## Resumen

| Concepto | PokeAPI | Mastodon |
|---|---|---|
| URL base | `https://pokeapi.co/api/v2` | `https://<instancia>/api/v1` |
| Leer datos | `GET` sin registro | `GET` sin registro (endpoints públicos) |
| Formato | JSON | JSON (con HTML en `content`) |
| Paginación | `limit` y `offset` | `limit` y `max_id` |
| Escribir datos | No permite | `POST` con token en `Authorization` |

## Ejercicios

1. Escribe una función que reciba una lista de pokémon y regrese un diccionario `{nombre_en_español: nombre_en_japonés}`.
2. Elige una etiqueta en español (por ejemplo `#libros` o `#méxico`), junta al menos 100 publicaciones y calcula las 20 palabras más frecuentes, excluyendo palabras vacías (artículos, preposiciones, etc.).
3. Compara la proporción tipo/token de publicaciones en dos idiomas distintos. ¿Qué factores podrían explicar la diferencia?
4. Usa las funciones del cuaderno `03_analisis_de_enunciados_y_palabras.ipynb` para separar las publicaciones en enunciados y contar sílabas.